In [3]:
!pip install openai

from google.colab import userdata
from openai import OpenAI
import json
import os
from datetime import datetime

client = OpenAI(api_key=userdata.get("Open_AI_Key"))

TASKS_FILE = "tasks.json"

def parse_task(user_input):
    today = datetime.now().strftime("%Y-%m-%d")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""오늘 날짜는 {today}이야.
다음 문장에서 할일 정보를 추출해서 JSON으로만 답해줘:
"{user_input}"

형식:
{{"title": "할일 내용", "date": "YYYY-MM-DD", "time": "HH:MM"}}

날짜/시간 정보가 없으면 null로 해줘."""
        }]
    )

    result = response.choices[0].message.content
    result = result.strip().strip("```json").strip("```").strip()
    return json.loads(result)

def load_tasks():
    if not os.path.exists(TASKS_FILE):
        return []
    with open(TASKS_FILE, "r") as f:
        return json.load(f)

def save_tasks(tasks):
    with open(TASKS_FILE, "w") as f:
        json.dump(tasks, f, ensure_ascii=False, indent=2)

def add_task(user_input):
    task = parse_task(user_input)
    tasks = load_tasks()
    task["id"] = len(tasks) + 1
    tasks.append(task)
    save_tasks(tasks)
    time_str = task['time'] if task['time'] else "시간 미정"
    print(f"✅ 추가됨: {task['title']} | {task['date']} {time_str}")

def list_tasks():
    tasks = load_tasks()
    if not tasks:
        print("할일이 없어요!")
        return
    for t in tasks:
        time_str = t['time'] if t['time'] else "시간 미정"
        print(f"[{t['id']}] {t['title']} | {t['date']} {time_str}")

def delete_task(task_id):
    tasks = load_tasks()
    tasks = [t for t in tasks if t["id"] != task_id]
    save_tasks(tasks)
    print(f"🗑️ {task_id}번 삭제됐어요!")

def main():
    print("📝 AI 할일 관리자 (종료: quit)")
    print("명령어: add [할일] / list / delete [번호]")

    while True:
        cmd = input("\n> ").strip()

        if cmd == "quit":
            print("종료할게요!")
            break
        elif cmd == "list":
            list_tasks()
        elif cmd.startswith("delete "):
            try:
                task_id = int(cmd.split()[1])
                delete_task(task_id)
            except:
                print("예시: delete 1")
        elif cmd.startswith("add "):
            user_input = cmd[4:]
            add_task(user_input)
        else:
            print("명령어: add [할일] / list / delete [번호]")

main()


📝 AI 할일 관리자 (종료: quit)
명령어: add [할일] / list / delete [번호]

> list
할일이 없어요!

> add 내일 모레 9시에 운동
✅ 추가됨: 운동 | 2026-05-09 09:00

> add 내일 오후 5시 약속
✅ 추가됨: 약속 | 2026-05-08 17:00

> add 내일 오후 1시는 공부
✅ 추가됨: 공부 | 2026-05-08 13:00

> list
[1] 운동 | 2026-05-09 09:00
[2] 약속 | 2026-05-08 17:00
[3] 공부 | 2026-05-08 13:00

> delete 1
🗑️ 1번 삭제됐어요!

> list
[2] 약속 | 2026-05-08 17:00
[3] 공부 | 2026-05-08 13:00

> quit
종료할게요!
